
# Lusso & Risaliti 2016: α_OX – L_UV relation for AGN discs

The ultraviolet-to-X-ray spectral slope α_OX encodes the fundamental
physics of accretion discs. At higher bolometric luminosities, discs
shift toward cooler effective temperatures and steeper UV slopes,
reducing the X-ray-to-UV flux ratio. We compute α_OX for 15 tengri AGN
disc models (multicolor, no torus/lines) across log L_bol ∈ [10.5, 14.0],
measuring at rest-frame 2500 Å (UV) and 2 keV (X-ray). The Lusso &
Risaliti 2016 fit α_OX = −0.166 log L_2500 + 4.74 captures the
observational trend that luminous quasars are more UV-bright and
X-ray-weak.

Reference: Lusso & Risaliti 2016, ApJ, 819, 154
(α_OX–L_UV relation); Tananbaum 1979, ApJ, 234, L9 (definition of α_OX).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

# Build minimal AGN disc model (composable: disc only, no torus/lines/feii)
ssp = tengri.load_ssp()
model = tengri.SEDModel.build(
    ssp,
    sfh={"type": "dpl", "all_params": tengri.FIXED, "tau_gyr": 3.0, "log_total_mass": 10.0},
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.05, "tau_bc": 0.05},
    agn={
        "type": "composable",
        "disc": {"type": "multicolor", "all_params": tengri.FIXED},
        "all_params": tengri.FIXED,
        "lum_ratio": 1.0,  # Full AGN contribution
    },
    redshift=tengri.Fixed(0.0),  # Rest-frame only
)

baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

# Sweep log_lbol: 15 points from 10.5 to 14.0 L_sun
log_lbol_values = np.linspace(10.5, 14.0, 15)

# Reference wavelengths for α_OX calculation
lambda_uv_rest = 2500.0  # Å (rest-frame)
lambda_xray_rest = 2.0 * 2.418e-8 * 1e10  # 2 keV in Å (λ = hc/E)

# Frequency at reference wavelengths (Hz)
c_angstrom_per_s = 2.998e18  # c in Å/s
nu_uv = c_angstrom_per_s / lambda_uv_rest
nu_xray = c_angstrom_per_s / lambda_xray_rest


# Lusso & Risaliti 2016 fit (Eq. 1)
def lusso_risaliti_fit(log_l_2500):
    """α_OX = −0.166 log L_2500 + 4.74 (in cgs, L in erg/s)."""
    return -0.166 * log_l_2500 + 4.74


# Compute α_OX for each luminosity
alpha_ox_values = []
log_l_2500_values = []

for log_lbol in log_lbol_values:
    params = {**baseline, "agn_log_lbol": jnp.float64(log_lbol)}
    out = model.predict(params)

    wave = np.asarray(model.wavelengths)
    sed = np.asarray(out.rest_sed())  # erg/s/Hz

    # Linear interpolation to get L_ν at reference wavelengths
    # sed is L_ν(λ) in erg/s/Hz
    # For photometry at a single wavelength, we use the interpolated value directly
    l_nu_uv = np.interp(lambda_uv_rest, wave, sed, left=np.nan, right=np.nan)
    l_nu_xray = np.interp(lambda_xray_rest, wave, sed, left=np.nan, right=np.nan)

    # Convert L_ν to L (flux × 4π steradians, but for SED shape, just track the ratio)
    # For luminosity, we need the monochromatic luminosity L_ν at the reference wavelengths
    # α_OX = −log₁₀(L_2keV / L_2500) / log₁₀(ν_2keV / ν_2500)
    #      = −log₁₀(L_ν(2keV) / L_ν(2500Å)) / log₁₀(ν_2keV / ν_2500Å)
    # The 4π cancels; we use monochromatic luminosity L_ν

    if np.isfinite(l_nu_uv) and np.isfinite(l_nu_xray) and l_nu_uv > 0 and l_nu_xray > 0:
        # L_ν in erg/s/Hz → log L_2500 as log10(L_ν/erg/s/Hz) [for reference only]
        log_l_nu_2500 = np.log10(l_nu_uv)
        log_l_2500_values.append(log_l_nu_2500)

        # α_OX definition (Tananbaum 1979): power-law index connecting UV and X-ray
        # f_ν ∝ ν^(−α_OX), so α_OX = log(f_2keV / f_2500) / log(ν_2keV / ν_2500)
        # Since L_ν is flux × distance² (distance cancels in the ratio):
        alpha_ox = np.log10(l_nu_xray / l_nu_uv) / np.log10(nu_xray / nu_uv)
        alpha_ox_values.append(alpha_ox)

alpha_ox_values = np.array(alpha_ox_values)
log_l_2500_values = np.array(log_l_2500_values)

# Plot α_OX vs log L_2500 with Lusso & Risaliti 2016 fit overlay
fig, ax = plt.subplots(figsize=(7.0, 5.0))

# Scatter plot of computed values
ax.scatter(log_l_2500_values, alpha_ox_values, s=80, color="steelblue", alpha=0.7, zorder=3)

# Lusso & Risaliti 2016 fit
log_l_2500_fit = np.linspace(log_l_2500_values.min() - 0.5, log_l_2500_values.max() + 0.5, 100)
alpha_ox_fit = lusso_risaliti_fit(log_l_2500_fit)
ax.plot(log_l_2500_fit, alpha_ox_fit, "r--", lw=2.0, label=r"Lusso & Risaliti 2016", zorder=2)

ax.set_xlabel(r"$\log L_{2500} / (erg\ s^{-1}\ Hz^{-1})$", fontsize=12)
ax.set_ylabel(r"$\alpha_{\mathrm{OX}}$", fontsize=12)
ax.set_title("AGN Disc: UV-to-X-ray Spectral Slope vs. Luminosity", fontsize=13, pad=15)
ax.grid(True, alpha=0.3, linestyle=":", zorder=0)
ax.legend(loc="best", fontsize=11)

fig.tight_layout()
plt.savefig("plot_alpha_ox_lusso_risaliti.png", dpi=150, bbox_inches="tight")

print(f"α_OX values: min={alpha_ox_values.min():.3f}, max={alpha_ox_values.max():.3f}")
print(f"Log L_2500: min={log_l_2500_values.min():.2f}, max={log_l_2500_values.max():.2f}")